In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import spacy
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
import random

import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import spacy
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
import random

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
tiny_stories = load_dataset("roneneldan/TinyStories", split="train")

print(f"Total size: {len(tiny_stories)}")
print(f"Sample story: {tiny_stories[10]["text"]}")

model_name = "roneneldan/TinyStories-33M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, output_hidden_states=True)
model.eval()

print(model.config)

Total size: 2119719
Sample story: Once upon a time, there was a big car named Dependable. He had a very important job. Dependable would take a family to the park every day. The family had a mom, dad, and a little girl named Lily. They all had a lot of love for each other.

One day, when they got to the park, they saw a big sign that said, "Fun Race Today!" The family was very excited. They knew that Dependable was very fast and could win the race. So, they decided to join the race.

The race started, and Dependable went very fast. The other cars tried to catch up, but Dependable was too quick. In the end, Dependable won the race! The family was so happy and proud of their car. They knew that their love for each other and their trust in Dependable made them win the race. And from that day on, they had even more fun at the park, knowing that they had the fastest and most dependable car around.


[transformers] The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-33M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3}.attn.attention.masked_bias | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3}.attn.attention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPTNeoConfig {
  "activation_function": "gelu_new",
  "architectures": [
    "GPTNeoForCausalLM"
  ],
  "attention_dropout": 0,
  "attention_layers": [
    "global",
    "local",
    "global",
    "local"
  ],
  "attention_types": [
    [
      [
        "global",
        "local"
      ],
      2
    ]
  ],
  "bos_token_id": 50256,
  "classifier_dropout": 0.1,
  "dtype": "float32",
  "embed_dropout": 0,
  "eos_token_id": 50256,
  "gradient_checkpointing": false,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": null,
  "layer_norm_epsilon": 1e-05,
  "max_position_embeddings": 2048,
  "model_type": "gpt_neo",
  "num_heads": 16,
  "num_layers": 4,
  "output_hidden_states": true,
  "pad_token_id": null,
  "resid_dropout": 0,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.9.0",
  "use_cache": true,
  

In [4]:
N_STORIES = 500

sampled_stories = tiny_stories.shuffle(seed=SEED).select(range(N_STORIES))
print(f"Sampled {len(sampled_stories)} stories")
print(f"Example: {sampled_stories[0]['text'][:200]}\n")

lengths = [len(tokenizer(s["text"])["input_ids"]) for s in sampled_stories]
print(f"Mean tokens per story: {np.mean(lengths):.0f}")
print(f"Min: {min(lengths)}, Max: {max(lengths)}")
print(f"Total tokens: {sum(lengths)}")
print(f"Estimated noun/verb tokens: {sum(lengths) // 2}")

Sampled 500 stories
Example: Tim and Mia like to play in the park. They see a big club on the ground. It is brown and long and heavy.

"Look, a club!" Tim says. "I can lift it!"

He tries to lift the club, but it is too tough. He

Mean tokens per story: 225
Min: 89, Max: 1022
Total tokens: 112638
Estimated noun/verb tokens: 56319


In [7]:
# run this in terminal first: python -m spacy download en_core_web_sm

nlp = spacy.load("en_core_web_sm")

def get_pos_tags(text):
    doc = nlp(text)
    # return list of (word, POS tag) pairs
    return [(token.text, token.pos_) for token in doc]

# test on one story
example_tags = get_pos_tags(sampled_stories[0]["text"])
print(example_tags[:20])

[('Tim', 'PROPN'), ('and', 'CCONJ'), ('Mia', 'PROPN'), ('like', 'VERB'), ('to', 'PART'), ('play', 'VERB'), ('in', 'ADP'), ('the', 'DET'), ('park', 'NOUN'), ('.', 'PUNCT'), ('They', 'PRON'), ('see', 'VERB'), ('a', 'DET'), ('big', 'ADJ'), ('club', 'NOUN'), ('on', 'ADP'), ('the', 'DET'), ('ground', 'NOUN'), ('.', 'PUNCT'), ('It', 'PRON')]


In [8]:
pos_tagged_stories = []

for story in sampled_stories:
    doc = nlp(story["text"])
    tokens_and_tags = [(token.text, token.pos_) for token in doc]
    pos_tagged_stories.append(tokens_and_tags)

noun_count = sum(1 for story in pos_tagged_stories 
                 for _, tag in story if tag == "NOUN")
verb_count = sum(1 for story in pos_tagged_stories 
                 for _, tag in story if tag == "VERB")

print(f"Total NOUN tokens: {noun_count}")
print(f"Total VERB tokens: {verb_count}")
print(f"Ratio: {noun_count/verb_count:.2f}")

Total NOUN tokens: 13714
Total VERB tokens: 16479
Ratio: 0.83


In [ ]:
import random

all_tagged_tokens = [
    (story_idx, token_idx, word, tag)
    for story_idx, story in enumerate(pos_tagged_stories)
    for token_idx, (word, tag) in enumerate(story)
    if tag in ("NOUN", "VERB")
]

random.shuffle(all_tagged_tokens)

split = int(0.8 * len(all_tagged_tokens))
train_pool = all_tagged_tokens[:split]
test_set = all_tagged_tokens[split:]

print(f"Train pool: {len(train_pool)} tokens")
print(f"Test set: {len(test_set)} tokens")

In [ ]:
# Build per-word examples: (residual vectors at each layer, label)
# label: 0 = NOUN, 1 = VERB

NUM_LAYERS = model.config.num_layers + 1

layer_activations = {layer: [] for layer in range(NUM_LAYERS)}
labels = []
meta = []  # (story_idx, word, tag) for debugging

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

for story_idx, story in enumerate(sampled_stories):
    text = story["text"]

    doc = nlp(text)
    target_words = [
        (tok.idx, tok.idx + len(tok.text), tok.pos_)
        for tok in doc
        if tok.pos_ in ("NOUN", "VERB")
    ]
    if not target_words:
        continue

    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    )
    offsets = enc["offset_mapping"][0].tolist()  # list of [start, end] per BPE token
    input_ids = {k: v.to(device) for k, v in enc.items() if k != "offset_mapping"}

    with torch.no_grad():
        out = model(**input_ids)
    hidden = out.hidden_states  # tuple: (NUM_LAYERS) each [1, seq_len, 768]

    # for each target word, find the FIRST bpe token whose span starts inside it
    for w_start, w_end, pos in target_words:
        bpe_pos = None
        for i, (b_start, b_end) in enumerate(offsets):
            if b_start == b_end:  # special tokens have (0,0)
                continue
            if b_start >= w_start and b_start < w_end:
                bpe_pos = i
                break
        if bpe_pos is None:
            continue 

        labels.append(0 if pos == "NOUN" else 1)
        meta.append((story_idx, text[w_start:w_end], pos))
        for layer in range(NUM_LAYERS):
            vec = hidden[layer][0, bpe_pos].cpu().numpy()
            layer_activations[layer].append(vec)

# convert to arrays
import numpy as np
X_by_layer = {layer: np.array(layer_activations[layer]) for layer in range(NUM_LAYERS)}
y = np.array(labels)

print(f"Total labeled examples: {len(y)}")
print(f"NOUN: {(y==0).sum()}, VERB: {(y==1).sum()}")
print(f"Activation shape per layer: {X_by_layer[0].shape}")  # (N, 768)
print(f"Number of layers (incl. embedding): {NUM_LAYERS}")

In [ ]:
from sklearn.model_selection import train_test_split

X_train_by_layer, X_test_by_layer, y_train, y_test = {}, {}, None, None

for layer in range(NUM_LAYERS):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_by_layer[layer], y, test_size=0.2, random_state=SEED, stratify=y
    )
    X_train_by_layer[layer] = X_tr
    X_test_by_layer[layer] = X_te
    if y_train is None:
        y_train, y_test = y_tr, y_te

majority_baseline = max((y_test == 0).mean(), (y_test == 1).mean())
print(f"Train size: {len(y_train)}, Test size: {len(y_test)}")
print(f"Test class balance — NOUN: {(y_test==0).mean():.2%}, VERB: {(y_test==1).mean():.2%}")
print(f"Majority class baseline accuracy: {majority_baseline:.4f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

results = {"layer": [], "accuracy": [], "auroc": []}

for layer in range(NUM_LAYERS):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train_by_layer[layer])
    X_te = scaler.transform(X_test_by_layer[layer])

    probe = LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000, random_state=SEED)
    probe.fit(X_tr, y_train)

    y_pred = probe.predict(X_te)
    y_prob = probe.predict_proba(X_te)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    results["layer"].append(layer)
    results["accuracy"].append(acc)
    results["auroc"].append(auc)

    print(f"Layer {layer:2d} | Accuracy: {acc:.4f} | AUROC: {auc:.4f}")

print(f"\nMajority baseline: {majority_baseline:.4f}")

In [ ]:
layers = results["layer"]
accuracies = results["accuracy"]
aurocs = results["auroc"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(layers, accuracies, marker="o", color="steelblue", linewidth=2, label="Probe accuracy")
ax1.axhline(majority_baseline, color="gray", linestyle="--", linewidth=1.5, label=f"Majority baseline ({majority_baseline:.2%})")
ax1.set_xlabel("Layer")
ax1.set_ylabel("Accuracy")
ax1.set_title("Linear Probe Accuracy by Layer\n(TinyStories-33M, NOUN vs. VERB)")
ax1.set_xticks(layers)
ax1.set_ylim(0.5, 1.02)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(layers, aurocs, marker="o", color="darkorange", linewidth=2, label="AUROC")
ax2.axhline(0.5, color="gray", linestyle="--", linewidth=1.5, label="Random baseline (0.50)")
ax2.set_xlabel("Layer")
ax2.set_ylabel("AUROC")
ax2.set_title("Linear Probe AUROC by Layer\n(TinyStories-33M, NOUN vs. VERB)")
ax2.set_xticks(layers)
ax2.set_ylim(0.45, 1.02)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("probe_results.png", dpi=150, bbox_inches="tight")
plt.show()

best_layer = layers[accuracies.index(max(accuracies))]
first_strong_layer = next(l for l, a in zip(layers, accuracies) if a > majority_baseline + 0.05)
print(f"Best layer: {best_layer} (accuracy {max(accuracies):.4f})")
print(f"First layer beating baseline by >5pp: Layer {first_strong_layer}")